# 1 Join All Scraped Sources
Enrich the OCR truck-stop dataset by joining coordinate / detail data from
three scraped sources (truckstopsandservices.com, Yelp, Yellow Pages) using
previously-computed phone-match row-ID columns.

Inputs:
- data/1_derived/4_scrape_yellowpages/1_yellowpages_phone_matched.csv  (main OCR dataset — final yellowpages pipeline output)
- output/0_raw/1_truckstops_services_webscraped.csv  (truckstopsandservices.com scraped data)
- output/0_raw/2_yelp_webscraped.csv  (Yelp scraped data)
- output/0_raw/3_yellowpages_webscraped.csv  (Yellow Pages scraped data)

Output: data/1_derived/5_geocode_truck_stops/1_joined_all_sources.csv

In [1]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
OUT_DIR = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_FILE      = PROJECT_ROOT / 'data' / '1_derived' / '4_scrape_yellowpages' / '1_yellowpages_phone_matched.csv'
WEBSCRAPED_FILE = PROJECT_ROOT / 'output' / '0_raw' / '1_truckstops_services_webscraped.csv'
YELP_FILE       = PROJECT_ROOT / 'output' / '0_raw' / '2_yelp_webscraped.csv'
YP_FILE         = PROJECT_ROOT / 'output' / '0_raw' / '3_yellowpages_webscraped.csv'
OUT_FILE        = OUT_DIR / '1_joined_all_sources.csv'

df = pd.read_csv(MAIN_FILE, low_memory=False)
webscraped_df = pd.read_csv(WEBSCRAPED_FILE, low_memory=False)
yelp_df = pd.read_csv(YELP_FILE, low_memory=False)
yellowpages_df = pd.read_csv(YP_FILE, low_memory=False)

print(f'Main dataset      : {len(df):,} rows')
print(f'Webscraped dataset: {len(webscraped_df):,} rows')
print(f'Yelp dataset      : {len(yelp_df):,} rows')
print(f'YellowPages dataset: {len(yellowpages_df):,} rows')

Main dataset      : 38,135 rows
Webscraped dataset: 15,526 rows
Yelp dataset      : 15,689 rows
YellowPages dataset: 10,533 rows


In [ ]:
# Join truckstopsandservices.com data using phone_scraped_matches_row_ids
# These IDs may be stored as:
#   - numpy-style arrays: '[10517  2750]' (spaces, no commas)
#   - Python list strings: '[10535, 2767]' (with commas)

def parse_index_list(val):
    """Parse row-ID list stored as either numpy-style or Python-style string."""
    s = str(val).strip()
    if s in ('', '[]', 'nan'):
        return []
    # Try Python literal first (handles commas)
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [int(x) for x in result]
    except Exception:
        pass
    # Fallback: numpy-style (space-separated, no commas)
    s = s.strip('[]').strip()
    if not s:
        return []
    return [int(x) for x in s.replace(',', ' ').split()]


def get_webscraped_data(row):
    try:
        idx_list = parse_index_list(row['phone_scraped_matches_row_ids'])
        if not idx_list:
            return pd.Series()
        selected = webscraped_df.iloc[idx_list]
        prefixed = {}
        for col in webscraped_df.columns:
            values = [str(v) for v in selected[col].tolist()]
            prefixed['Webscraped_Phone_' + col] = '; '.join(values)
        return pd.Series(prefixed)
    except Exception:
        return pd.Series()


print('Joining webscraped data by phone match IDs...')
webscraped_cols = df.apply(get_webscraped_data, axis=1)
df = pd.concat([df, webscraped_cols], axis=1)
print(f'  Added {len(webscraped_cols.columns)} Webscraped_Phone_ columns')

In [ ]:
# Compute Webscraped_Place_Match
# Multi-criteria place validation: union(ZIP, State) ∩ union(City, Exit) ∩ Road ∩ Label

def parse_list(val):
    """Parse a list stored as either Python list string or numpy-style array."""
    s = str(val).strip()
    if s in ('', '[]', 'nan', 'None'):
        return []
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [int(x) for x in result]
    except Exception:
        pass
    # Fallback: numpy-style (space-separated)
    s = s.strip('[]').strip()
    if not s:
        return []
    try:
        return [int(x) for x in s.replace(',', ' ').split()]
    except Exception:
        return []


def get_place_match(row):
    zip_list   = set(parse_list(row.get('ZIP_scraped_matches_row_ids', [])))
    state_list = set(parse_list(row.get('State_scraped_matches_row_ids', [])))
    city_list  = set(parse_list(row.get('City_scraped_matches_row_ids', [])))
    exit_list  = set(parse_list(row.get('Exit_scraped_matches_row_ids', [])))
    road_list  = set(parse_list(row.get('Road_scraped_matches_row_ids', [])))
    label_list = set(parse_list(row.get('Label_scraped_matches_row_ids', [])))
    # Union ZIP and State
    base = zip_list.union(state_list)
    # Intersect with City or Exit
    city_exit = city_list.union(exit_list)
    filtered = base.intersection(city_exit)
    # Keep only those found in both Road and Label
    final = [x for x in filtered if x in road_list and x in label_list]
    return final


print('Computing Webscraped_Place_Match...')
df['Webscraped_Place_Match'] = df.apply(get_place_match, axis=1)
matched = (df['Webscraped_Place_Match'].apply(len) > 0).sum()
print(f'  Rows with a place match: {matched:,} ({matched/len(df)*100:.1f}%)')

In [4]:
# Join truckstopsandservices.com data using Webscraped_Place_Match IDs
# Prefix columns with 'Webscraped_PlacedMatched_'

def get_webscraped_place_matched_data(row):
    try:
        idx_list = row['Webscraped_Place_Match']
        if not isinstance(idx_list, list) or not idx_list:
            return pd.Series()
        selected = webscraped_df.iloc[idx_list]
        prefixed = {}
        for col in webscraped_df.columns:
            values = [str(v) for v in selected[col].tolist()]
            prefixed['Webscraped_PlacedMatched_' + col] = '; '.join(values)
        return pd.Series(prefixed)
    except Exception:
        return pd.Series()


print('Joining webscraped data by place match IDs...')
webscraped_place_cols = df.apply(get_webscraped_place_matched_data, axis=1)
df = pd.concat([df, webscraped_place_cols], axis=1)
print(f'  Added {len(webscraped_place_cols.columns)} Webscraped_PlacedMatched_ columns')

Joining webscraped data by place match IDs...


  Added 0 Webscraped_PlacedMatched_ columns


In [5]:
# Join Yelp data using Phone_Yelp_matches_row_ids
# These IDs are stored as Python list strings: '[1, 2, 3]'

def get_yelp_data(row):
    try:
        idx_list = row['Phone_Yelp_matches_row_ids']
        if isinstance(idx_list, str):
            try:
                idx_list = ast.literal_eval(idx_list)
            except Exception:
                return pd.Series()
        if not isinstance(idx_list, list) or not idx_list:
            return pd.Series()
        selected = yelp_df.iloc[idx_list]
        prefixed = {}
        for col in yelp_df.columns:
            values = [str(v) for v in selected[col].tolist()]
            prefixed['Yelp_' + col] = '; '.join(values)
        return pd.Series(prefixed)
    except Exception:
        return pd.Series()


print('Joining Yelp data by phone match IDs...')
yelp_cols = df.apply(get_yelp_data, axis=1)
df = pd.concat([df, yelp_cols], axis=1)
print(f'  Added {len(yelp_cols.columns)} Yelp_ columns')

Joining Yelp data by phone match IDs...


  Added 15 Yelp_ columns


In [6]:
# Join Yellow Pages data using Phone_Yellowpages_matches_row_ids
# These IDs are stored as Python list strings: '[1, 2, 3]'

def get_yellowpages_data(row):
    try:
        idx_list = row['Phone_Yellowpages_matches_row_ids']
        if isinstance(idx_list, str):
            try:
                idx_list = ast.literal_eval(idx_list)
            except Exception:
                return pd.Series()
        if not isinstance(idx_list, list) or not idx_list:
            return pd.Series()
        selected = yellowpages_df.iloc[idx_list]
        prefixed = {}
        for col in yellowpages_df.columns:
            values = [str(v) for v in selected[col].tolist()]
            prefixed['YellowPages_' + col] = '; '.join(values)
        return pd.Series(prefixed)
    except Exception:
        return pd.Series()


print('Joining YellowPages data by phone match IDs...')
yellowpages_cols = df.apply(get_yellowpages_data, axis=1)
df = pd.concat([df, yellowpages_cols], axis=1)
print(f'  Added {len(yellowpages_cols.columns)} YellowPages_ columns')

Joining YellowPages data by phone match IDs...


  Added 21 YellowPages_ columns


In [7]:
# Save enriched dataset
df.to_csv(OUT_FILE, index=False)
print(f'Saved: {OUT_FILE}')
print(f'Final shape: {df.shape}')
df.head(3)

Saved: C:\Users\Owner\Desktop\Geocoding_Truck_Stops\data\1_derived\5_geocode_truck_stops\1_joined_all_sources.csv
Final shape: (38135, 129)


,clean_line1,clean_line2,line3,city,zip_code,label,phone,year,major_city,state,...,YellowPages_JSONLD_PHONE_1,YellowPages_JSONLD_STATE_1,YellowPages_JSONLD_STREET_1,YellowPages_JSONLD_ZIP_1,YellowPages_ORIGINAL_PHONE,YellowPages_PHONE,YellowPages_SCRAPED_AT,YellowPages_SEARCH_URL,YellowPages_STATUS,YellowPages_WEBSITE
0,"Blandford , 01008 Blandford Plaza EB Exxon # 5020",413-848-2056 I-90 ( MATP ) MM 29 EB,<U+25A1> <U+2610>,Blandford,1008,Blandford Plaza EB Exxon # 5020,413-848-2056,2006,Blandford,MA,...,nan,nan,nan,nan,413-848-2056,nan,2026-02-11T07:34:39.090Z,https://www.yellowpages.com/search?search_term...,no_results_found,nan
1,"Blandford , 01008 Blandford Plaza EB Exxon # 5020",413-848-2056 I-90 ( MATP ) MM 29 EB,24 HRS S,Blandford,1008,Blandford Plaza EB Exxon # 5020,413-848-2056,2007,Blandford,MA,...,nan,nan,nan,nan,413-848-2056,nan,2026-02-11T07:34:39.090Z,https://www.yellowpages.com/search?search_term...,no_results_found,nan
2,"Blandford , 01008 Blandford Plaza EB Exxon # 5020",413-848-2056 I-90 ( MATP ) MM 29 EB,HAS 24 SO <U+2610> <U+2610>,Blandford,1008,Blandford Plaza EB Exxon # 5020,413-848-2056,2008,Blandford,MA,...,nan,nan,nan,nan,413-848-2056,nan,2026-02-11T07:34:39.090Z,https://www.yellowpages.com/search?search_term...,no_results_found,nan


In [8]:
# Match condition analysis
enhanced_combined_condition = (
    ((df['Scraped_phone_match_rate'] == True) |
     (df['Scraped_zipcode_to_label_match_rate'] == '6/6 successful match') |
     (df['Scraped_zipcode_to_label_match_rate'] == '7/6 successful match')) |
    (df['Yelp_phone_match_rate'] == True) |
    (df['Yellowbook_phone_match_rate'] == True)
)

print('=== ENHANCED COMBINED CONDITION ANALYSIS ===')
print(f'Total rows in dataset: {len(df)}')

enhanced_success_count = enhanced_combined_condition.sum()
print(f'Rows meeting enhanced criteria: {enhanced_success_count} ({enhanced_success_count/len(df)*100:.2f}%)')

enhanced_low_success_count = (~enhanced_combined_condition).sum()
print(f'Rows NOT meeting enhanced criteria: {enhanced_low_success_count} ({enhanced_low_success_count/len(df)*100:.2f}%)')

=== ENHANCED COMBINED CONDITION ANALYSIS ===
Total rows in dataset: 38135
Rows meeting enhanced criteria: 34305 (89.96%)
Rows NOT meeting enhanced criteria: 3830 (10.04%)


In [9]:
# Match condition vs coordinate availability
exclude = {'nan', '', 'None', 'removed'}
has_coord = lambda col: df[col].apply(lambda x: str(x).strip() not in exclude)

print('=== MATCH CONDITION vs COORDINATE AVAILABILITY ===\n')

checks = [
    ('Scraped_phone_match_rate == True',
     df['Scraped_phone_match_rate'] == True,
     'Webscraped_Phone_LD_Latitude', has_coord('Webscraped_Phone_LD_Latitude')),

    ("Scraped_zipcode_to_label_match_rate == '6/6 successful match'",
     df['Scraped_zipcode_to_label_match_rate'] == '6/6 successful match',
     'Webscraped_Phone_LD_Latitude', has_coord('Webscraped_Phone_LD_Latitude')),

    ("Scraped_zipcode_to_label_match_rate == '7/6 successful match'",
     df['Scraped_zipcode_to_label_match_rate'] == '7/6 successful match',
     'Webscraped_Phone_LD_Latitude', has_coord('Webscraped_Phone_LD_Latitude')),

    ('Yelp_phone_match_rate == True',
     df['Yelp_phone_match_rate'] == True,
     'Yelp_Latitude', has_coord('Yelp_Latitude')),

    ('Yellowbook_phone_match_rate == True',
     df['Yellowbook_phone_match_rate'] == True,
     'YellowPages_JSONLD_LAT_1', has_coord('YellowPages_JSONLD_LAT_1')),
]

for label, match_mask, coord_col, coord_mask in checks:
    n_match = match_mask.sum()
    n_both  = (match_mask & coord_mask).sum()
    n_missing_coord = (match_mask & ~coord_mask).sum()
    pct = n_both / n_match * 100 if n_match > 0 else 0
    print(f'  Condition : {label}')
    print(f'  Rows matching          : {n_match}')
    print(f'  Have coord ({coord_col}): {n_both} ({pct:.1f}%)')
    print(f'  Match but NO coord     : {n_missing_coord} ({100-pct:.1f}%)')
    print()

=== MATCH CONDITION vs COORDINATE AVAILABILITY ===

  Condition : Scraped_phone_match_rate == True
  Rows matching          : 29206
  Have coord (Webscraped_Phone_LD_Latitude): 29206 (100.0%)
  Match but NO coord     : 0 (0.0%)

  Condition : Scraped_zipcode_to_label_match_rate == '6/6 successful match'
  Rows matching          : 3947
  Have coord (Webscraped_Phone_LD_Latitude): 3544 (89.8%)
  Match but NO coord     : 403 (10.2%)

  Condition : Scraped_zipcode_to_label_match_rate == '7/6 successful match'
  Rows matching          : 23111
  Have coord (Webscraped_Phone_LD_Latitude): 20494 (88.7%)
  Match but NO coord     : 2617 (11.3%)

  Condition : Yelp_phone_match_rate == True
  Rows matching          : 21985
  Have coord (Yelp_Latitude): 21985 (100.0%)
  Match but NO coord     : 0 (0.0%)

  Condition : Yellowbook_phone_match_rate == True
  Rows matching          : 4951
  Have coord (YellowPages_JSONLD_LAT_1): 4905 (99.1%)
  Match but NO coord     : 46 (0.9%)

